In [1]:
"""
exp01 Step 02: Single-Feature Signal Check (stopping rule, PROTOCOL.md §13.1; base-rate version since protocol 1.0)

Question: does any single feature move the take profit rate AWAY FROM THE BASE RATE (the unconditional TP rate of the same
window at the same SL/TP distance), reliably, on training AND on the pooled validation quarters?

Changed from the v1 check (signal_check_v1, 2026-10-01; see docs/history): v1 compared bins with the BREAK-EVEN TP rate,
and market drift alone lifted 1,339 training bins above it. Now:
    - reference = the window's base rate per delta (two-sided: below the base rate = "do not enter here");
    - validation = the 4 most recent validation quarters POOLED; training = the 10-year window of the first pooled fold;
    - a bin's pass counts only if its rows cover at least 6 calendar months in each window;
    - the break-even TP rate is still reported (an above-base-rate bin can still be untradable after costs).

Per (feature, delta, bin): TP rate with a block-bootstrap interval (whole ISO weeks resampled), base rate, break-even rate,
mean net return, month count. The verdict reports how many training passes (5% of the tests) and how many signals
(tests x 2 x 2.5%^2) are expected by chance, and how many distinct feature bins the signals come from.
Verdict: STOP or pivot if no bin is a signal. CAUTION: with ~3,000 correlated tests about 4 signals are expected by chance;
a synthetic random walk produced 2 (PROTOCOL.md section 13). A CONTINUE that is not clearly above the chance count is
not evidence of a signal. One signal-check entry is written to the trial log.
Prerequisite: exp01 step 01. Outputs: store04_experiments/exp01_minute_entry/step02_signal_check_data/.
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT PLOTLY
import plotly.express as px
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp01_minute_entry import config as exp_config
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import get_date_range_file_path_list, write_csv_file_to_path
# IMPORT BARRIER LABEL AND TRADE EXECUTION FUNCTIONS
from so.features.barrier_labels import get_delta_col_str
from so.core.trade_execution import get_breakeven_tp_rate_arr
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp01_minute_entry.model_dataset import read_model_dataset_pdf, get_model_feature_col_str_list
from experiments.exp01_minute_entry.signal_check import get_signal_check_pdf, get_signal_check_verdict_dict
from experiments.exp01_minute_entry.walk_forward import get_fold_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_signal_check_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "signal_check")

Experiment:	exp01_minute_entry	Protocol:	1.0	Config Hash:	16493e598a
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step02_signal_check_data/
ℹ️ No earlier 'signal_check' entry of exp01_minute_entry in the trial log (config hash 16493e598a).


In [4]:
# DEFINE WHETHER ABSOLUTE-DOLLAR FEATURES ARE INCLUDED (THEIR BINS MOSTLY SEPARATE YEARS / PRICE LEVELS UNTIL SCALING IS DESIGNED)
INCLUDE_ABSOLUTE_FEATURES_BOOL = False
# COLLECT THE SESSION DATES FROM THE MODEL DATASET FILES
dataset_file_path_list = get_date_range_file_path_list(paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_model_dataset_data"))
session_date_list = [pd.to_datetime(f.split("_")[-1].replace(".csv", "")).date() for f in dataset_file_path_list]
# CALL FUNCTION TO GET THE FOLDS (SIGNAL CHECK TRAINING WINDOW ONLY) AND COLLECT THE POOLED FOLDS
fold_pdf = get_fold_pdf(session_date_list, train_window_years_list_in=[exp_config.SIGNAL_CHECK_TRAIN_WINDOW_YEARS])
pooled_fold_pdf = fold_pdf.iloc[-exp_config.SIGNAL_CHECK_POOLED_QUARTER_COUNT:]
first_pooled_fold_row = pooled_fold_pdf.iloc[0]
# DEFINE THE TRAINING WINDOW (10 YEARS, ENDING AN EMBARGO BEFORE THE FIRST POOLED VALIDATION QUARTER)
train_start, train_end = first_pooled_fold_row[f"train_start_{exp_config.SIGNAL_CHECK_TRAIN_WINDOW_YEARS}y"], first_pooled_fold_row["train_end"]
# DISPLAY INFORMATION
print(f"Train:\t\t{train_start} -> {train_end}")
for _, row in pooled_fold_pdf.iterrows():
    print(f"Validation:\t{row['valid_start']} -> {row['valid_end']}\t(fold {row['fold_id']})")
print(f"Latest test window (not read):\t{fold_pdf['test_start'].iloc[-1]} -> {fold_pdf['test_end'].iloc[-1]}")

Train:		2015-04-16 -> 2025-04-15
Validation:	2025-05-01 -> 2025-07-30	(fold 41)
Validation:	2025-07-31 -> 2025-10-30	(fold 42)
Validation:	2025-10-31 -> 2026-01-30	(fold 43)
Validation:	2026-01-30 -> 2026-04-29	(fold 44)
Latest test window (not read):	2026-05-14 -> 2026-08-13


In [5]:
# CALL FUNCTION TO READ THE TRAINING ROWS (SAMPLED AS FOR TRAINING, TO LIMIT OVERLAPPING TRADES)
signal_train_pdf = read_model_dataset_pdf(str(train_start), str(train_end), exp_config.SIGNAL_CHECK_DELTA_LIST, exp_config.SAMPLING_MODE, exp_config.SAMPLE_EVERY_N_MINUTES)
# CALL FUNCTION TO READ THE POOLED VALIDATION ROWS (SAME SAMPLING; ADJACENT QUARTERS CAN SHARE A SESSION, DUPLICATES DROPPED)
signal_valid_pdf = pd.concat([read_model_dataset_pdf(str(row["valid_start"]), str(row["valid_end"]), exp_config.SIGNAL_CHECK_DELTA_LIST, exp_config.SAMPLING_MODE,
                                                     exp_config.SAMPLE_EVERY_N_MINUTES, alert_in=False) for _, row in pooled_fold_pdf.iterrows()], ignore_index=True) \
                     .drop_duplicates(subset=["decision_ts"]).reset_index(drop=True)
# DEFINE THE FEATURE LIST
signal_scale_type_list = exp_config.MODEL_FEATURE_SCALE_TYPE_LIST + (["absolute"] if INCLUDE_ABSOLUTE_FEATURES_BOOL else [])
signal_feature_col_str_list = get_model_feature_col_str_list(signal_train_pdf, signal_scale_type_list)
# DISPLAY INFORMATION
print(f"Train Rows:\t{len(signal_train_pdf):,}\tPooled Validation Rows:\t{len(signal_valid_pdf):,}")
print(f"Features ({len(signal_feature_col_str_list)}):\t{signal_feature_col_str_list}")

Reading 2,516 model dataset files (2015-04-16 -> 2025-04-15, sampling: 'interval')
Train Rows:	60,144	Pooled Validation Rows:	5,964
Features (33):	['trend', 'segment', 'color', 'bot_wick_pct', 'body_pct', 'top_wick_pct', 'stoch_k', 'stoch_d', 'rsi', 'bb_pct', 'dc_pct', 'ms_minima_trend', 'ms_maxima_trend', 'ms_low_status', 'ms_high_status', 'ms_trend', 'minima_count', 'maxima_count', 'prev_seg_delta_pct', 'seg_cs_count', 'seg', 'minutes_since_open', 'minutes_to_close', 'day_of_week', 'overnight_gap_pct', 'prev_day_return_pct', 'prev_day_range_pct', 'prev_day_high_dist_pct', 'prev_day_low_dist_pct', 'intraday_return_pct', 'daily_volatility', 'relative_volume', 'ath_drawdown_pct']


In [6]:
"""
Unconditional take profit rate per delta (no feature) = the BASE RATE every bin is compared with. Because SPY drifted
upwards, wide deltas held for days show TP rates above break-even without any signal: that is drift, not information.
"""
# LIST TO HOLD THE ROWS
unconditional_dict_list = []
# ITERATE OVER THE WINDOWS AND DELTAS
for window_str, window_pdf in [("train", signal_train_pdf), ("valid", signal_valid_pdf)]:
    for delta in exp_config.SIGNAL_CHECK_DELTA_LIST:
        # DEFINE THE LABEL COLUMNS
        delta_col_str = get_delta_col_str(delta)
        resolved_pdf = window_pdf[window_pdf[f"y_tp_{delta_col_str}"].notna()]
        # APPEND THE ROW
        unconditional_dict_list.append({"window": window_str, "delta": delta, "row_count": len(resolved_pdf),
                                        "base_tp_rate": resolved_pdf[f"y_tp_{delta_col_str}"].mean(),
                                        "breakeven_tp_rate": get_breakeven_tp_rate_arr(resolved_pdf["entry_price"].to_numpy(), delta).mean(),
                                        "mean_net_return": resolved_pdf[f"net_return_{delta_col_str}"].mean(),
                                        "tl_share": (resolved_pdf[f"exit_reason_{delta_col_str}"] == "TL").mean()})
# CONVERT TO A DATAFRAME
unconditional_pdf = pd.DataFrame(unconditional_dict_list)
# CREATE A FIGURE
fig = px.line(unconditional_pdf.melt(id_vars=["window", "delta"], value_vars=["base_tp_rate", "breakeven_tp_rate"]),
              x="delta", y="value", color="variable", line_dash="window", markers=True, log_x=True,
              title="Base (Unconditional) TP Rate vs Breakeven TP Rate", template="plotly_dark", width=1100, height=500)
# DISPLAY FIGURE
fig.show()
# PREVIEW DATAFRAME
unconditional_pdf

,window,delta,row_count,base_tp_rate,breakeven_tp_rate,mean_net_return,tl_share
0,train,0.0010,60144,0.498088,0.539465,-0.000079,0.000000
1,train,0.0011,60144,0.499235,0.535930,-0.000076,0.000000
2,train,0.0013,60144,0.501181,0.530476,-0.000069,0.000000
3,train,0.0014,60144,0.502827,0.528326,-0.000065,0.000000
4,train,0.0016,60144,0.506418,0.524825,-0.000048,0.000000
5,train,0.0018,60144,0.508480,0.522093,-0.000034,0.000000
6,train,0.0020,60144,0.511572,0.519903,-0.000012,0.000000
7,train,0.0022,60144,0.513484,0.518108,0.000006,0.000000
8,train,0.0025,60144,0.515812,0.515951,0.000034,0.000000
9,train,0.0028,60144,0.517209,0.514253,0.000053,0.000000


In [7]:
# CALL FUNCTION TO RUN THE SIGNAL CHECK (PROTOCOL SETTINGS FROM THE EXPERIMENT CONFIG)
signal_check_pdf = get_signal_check_pdf(signal_train_pdf, signal_valid_pdf, signal_feature_col_str_list, exp_config.SIGNAL_CHECK_DELTA_LIST)
# CALL FUNCTION TO GET THE VERDICT
signal_verdict_dict = get_signal_check_verdict_dict(signal_check_pdf)
# DISPLAY INFORMATION
for key_str, value in signal_verdict_dict.items():
    print(f"{key_str}:\t{value}")

Feature:	trend	[1/33]
Feature:	segment	[2/33]
Feature:	color	[3/33]
Feature:	bot_wick_pct	[4/33]
Feature:	body_pct	[5/33]
Feature:	top_wick_pct	[6/33]
Feature:	stoch_k	[7/33]
Feature:	stoch_d	[8/33]
Feature:	rsi	[9/33]
Feature:	bb_pct	[10/33]
Feature:	dc_pct	[11/33]
Feature:	ms_minima_trend	[12/33]
Feature:	ms_maxima_trend	[13/33]
Feature:	ms_low_status	[14/33]
Feature:	ms_high_status	[15/33]
Feature:	ms_trend	[16/33]
Feature:	minima_count	[17/33]
Feature:	maxima_count	[18/33]
Feature:	prev_seg_delta_pct	[19/33]
Feature:	seg_cs_count	[20/33]
Feature:	seg	[21/33]
Feature:	minutes_since_open	[22/33]
Feature:	minutes_to_close	[23/33]
Feature:	day_of_week	[24/33]
Feature:	overnight_gap_pct	[25/33]
Feature:	prev_day_return_pct	[26/33]
Feature:	prev_day_range_pct	[27/33]
Feature:	prev_day_high_dist_pct	[28/33]
Feature:	prev_day_low_dist_pct	[29/33]
Feature:	intraday_return_pct	[30/33]
Feature:	daily_volatility	[31/33]
Feature:	relative_volume	[32/33]
Feature:	ath_drawdown_pct	[33/33]
test_co

In [8]:
# SAVE THE SIGNAL CHECK RESULTS
write_csv_file_to_path(signal_check_pdf, f"{output_path_str}signal_check_data.csv", "W")
write_csv_file_to_path(unconditional_pdf, f"{output_path_str}base_rate_data.csv", "W")
# LOG THE TRIAL (THE LATEST TEST WINDOW WAS NOT READ)
signal_trial_dict = log_trial_dict(exp_config, "signal_check",
                                   {"train_years": exp_config.SIGNAL_CHECK_TRAIN_WINDOW_YEARS, "train_start": train_start, "train_end": train_end,
                                    "valid_fold_id_list": pooled_fold_pdf["fold_id"].tolist(), "feature_count": len(signal_feature_col_str_list),
                                    "delta_list": exp_config.SIGNAL_CHECK_DELTA_LIST, "include_absolute_features": INCLUDE_ABSOLUTE_FEATURES_BOOL,
                                    "reference": "base rate", "min_month_count": exp_config.SIGNAL_CHECK_MIN_MONTH_COUNT},
                                   signal_verdict_dict, False, pooled_fold_pdf.iloc[-1], note_str_in="exp01 step02 signal check (stopping rule, two-sided, base rate, month minimum)")
print(f"Logged 1 signal-check entry (config hash {get_config_hash_str(exp_config)})")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step02_signal_check_data/signal_check_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step02_signal_check_data/signal_check_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step02_signal_check_data/base_rate_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp01_minute_entry/step02_signal_check_data/base_rate_data.csv' Has Been Saved!
Logged 1 signal-check entry (config hash 16493e598a)


In [9]:
# CALCULATE THE LIFT OVER THE BASE RATE IN EACH WINDOW
signal_check_pdf["train_lift"] = signal_check_pdf["train_tp_rate"] - signal_check_pdf["train_base_rate"]
signal_check_pdf["valid_lift"] = signal_check_pdf["valid_tp_rate"] - signal_check_pdf["valid_base_rate"]
# COLLECT THE LARGEST ABSOLUTE TRAINING LIFT PER FEATURE AND DELTA (TESTED BINS)
tested_pdf = signal_check_pdf[signal_check_pdf["tested"]]
best_lift_pdf = tested_pdf.loc[tested_pdf.groupby(["feature", "delta"])["train_lift"].apply(lambda s: s.abs().idxmax())].pivot(index="feature", columns="delta", values="train_lift")
# CREATE A HEATMAP (SIGNED LIFT OF THE STRONGEST TRAINING BIN)
fig = px.imshow(best_lift_pdf, color_continuous_scale="RdYlGn", color_continuous_midpoint=0, aspect="auto",
                title="Strongest Training Lift (bin TP rate - base rate) Per Feature And Delta", template="plotly_dark", width=1100, height=900)
# DISPLAY FIGURE
fig.show()

In [10]:
# PREVIEW THE SIGNAL BINS (SAME SIDE ON TRAINING AND POOLED VALIDATION)
display(signal_check_pdf[signal_check_pdf["signal"]].sort_values("valid_lift", ascending=False))
# PREVIEW THE TESTED BINS WITH THE LARGEST TRAINING LIFT (BOTH SIDES)
tested_pdf.reindex(tested_pdf["train_lift"].abs().sort_values(ascending=False).index)[
    ["feature", "delta", "bin", "train_row_count", "train_month_count", "train_tp_rate", "train_tp_rate_ci_low", "train_tp_rate_ci_high", "train_base_rate",
     "valid_row_count", "valid_month_count", "valid_tp_rate", "valid_base_rate", "train_breakeven_tp_rate", "signal"]].head(30).round(4)

,feature,delta,bin,train_row_count,train_month_count,train_tp_rate,train_tp_rate_ci_low,train_tp_rate_ci_high,train_base_rate,train_breakeven_tp_rate,...,valid_net_return_ci_high,tested,train_above,train_below,valid_above,valid_below,signal,above_breakeven,train_lift,valid_lift
4927,intraday_return_pct,0.0014,"(0.00373, 0.00636]",6014,121,0.536581,0.512264,0.560018,0.502827,0.527571,...,0.000609,True,True,False,True,False,True,True,0.033755,0.100661
4917,intraday_return_pct,0.0013,"(0.00373, 0.00636]",6014,121,0.528766,0.505814,0.551665,0.501181,0.529664,...,0.000532,True,True,False,True,False,True,True,0.027586,0.092301
4937,intraday_return_pct,0.0016,"(0.00373, 0.00636]",6014,121,0.542401,0.516381,0.567325,0.506418,0.524162,...,0.000686,True,True,False,True,False,True,False,0.035983,0.085888
4321,prev_day_range_pct,0.0018,"(0.0152, 0.0208]",6000,78,0.488833,0.468888,0.507623,0.508480,0.521167,...,0.000275,True,False,True,False,True,True,False,-0.019646,-0.073105


,feature,delta,bin,train_row_count,train_month_count,train_tp_rate,train_tp_rate_ci_low,train_tp_rate_ci_high,train_base_rate,valid_row_count,valid_month_count,valid_tp_rate,valid_base_rate,train_breakeven_tp_rate,signal
5272,daily_volatility,0.0063,"(-inf, 0.00429]",6036,29,0.6178,0.5380,0.6982,0.5321,312.0,3.0,0.3429,0.4958,0.5075,False
5262,daily_volatility,0.0056,"(-inf, 0.00429]",6036,29,0.6148,0.5356,0.6915,0.5327,312.0,3.0,0.3526,0.5025,0.5085,False
5282,daily_volatility,0.0071,"(-inf, 0.00429]",6036,29,0.6059,0.5249,0.6899,0.5289,312.0,3.0,0.3013,0.4842,0.5067,False
5727,ath_drawdown_pct,0.0100,"(-0.00501, -0.00202]",6015,67,0.4336,0.3599,0.5170,0.5081,929.0,9.0,0.3380,0.4759,0.5040,False
5292,daily_volatility,0.0079,"(-inf, 0.00429]",6036,29,0.5982,0.5119,0.6890,0.5253,312.0,3.0,0.2340,0.4743,0.5060,False
5717,ath_drawdown_pct,0.0089,"(-0.00501, -0.00202]",6015,67,0.4449,0.3765,0.5241,0.5166,929.0,9.0,0.3143,0.4700,0.5045,False
5252,daily_volatility,0.0050,"(-inf, 0.00429]",6036,29,0.6011,0.5212,0.6766,0.5334,312.0,3.0,0.3814,0.5080,0.5095,False
5296,daily_volatility,0.0089,"(0.00429, 0.00527]",6000,39,0.4518,0.3662,0.5490,0.5166,456.0,4.0,0.5592,0.4700,0.5048,False
5242,daily_volatility,0.0045,"(-inf, 0.00429]",6036,29,0.5971,0.5219,0.6689,0.5326,312.0,3.0,0.3846,0.5027,0.5105,False
5726,ath_drawdown_pct,0.0100,"(-0.00922, -0.00501]",6014,69,0.4446,0.3699,0.5202,0.5081,925.0,10.0,0.5103,0.4759,0.5039,False


In [11]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,1,1,0,2026-10-05T15:17:38,2026-10-05T15:17:38
1,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
2,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
3,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
4,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
5,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
6,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
7,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
